In [1]:
# ---------------
# Dependencies
# ---------------

import os
import numpy
import pandas
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

In [2]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ----------------
# Configuration
# ----------------

DATASET = "/content/drive/MyDrive/data/dataset_original.csv"
NUM_API_CALLS = 307
SEQUENCE_LENGTH = 100
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
from sklearn.model_selection import train_test_split

df = pandas.read_csv(DATASET)

X = df.drop(columns=['hash','malware'])
y = df['malware']

def make_balanced(df):
    malware = df[df['malware'] == 1]
    benign = df[df['malware'] == 0]
    malware_down = malware.sample(len(benign), random_state=42)
    return pandas.concat([malware_down, benign]).sample(frac=1, random_state=42)

# Balanced
# df = make_balanced(df)

train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['malware'],
    random_state=42
)

In [5]:
# ----------
# Dataset
# ----------

class MalwareGraphDataset(Dataset):
    def __init__(self, df):
        self.sequences = df.drop(columns=['hash','malware']).values
        self.labels = torch.tensor(df['malware'].values, dtype=torch.float32)

    def seq_to_adj(self, seq):
        adj = torch.zeros((NUM_API_CALLS, NUM_API_CALLS))

        for i in range(len(seq)-1):
            src = seq[i]
            dst = seq[i+1]

            if 0 <= src < NUM_API_CALLS and 0 <= dst < NUM_API_CALLS:
                adj[src, dst] = 1

        return adj

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        adj = self.seq_to_adj(seq)

        X = F.one_hot(
            torch.tensor(seq, dtype=torch.long),
            num_classes=NUM_API_CALLS
        ).float().permute(1, 0)

        return adj, X, self.labels[idx]

    def __len__(self):
        return len(self.labels)

In [6]:
# ----------------------------
# Graph Convolutional Layer
# ----------------------------

class GraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(
            torch.randn(in_features, out_features) * 0.01
        )

    def forward(self, adj, X):
        B, N, _ = adj.size()
        I = torch.eye(N, device=adj.device).unsqueeze(0)
        A_hat = adj + I
        D = torch.sum(A_hat, dim=2)
        D_inv = torch.diag_embed(1.0 / (D + 1e-6))
        A_norm = D_inv @ A_hat
        Z = A_norm @ X
        Z = Z @ self.weight
        return Z

In [7]:
# --------
# Model
# --------

class DGCNN(nn.Module):
    def __init__(self, out_channels=31, dropout=0.6):
        super().__init__()
        self.gcn = GraphConvLayer(SEQUENCE_LENGTH, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(NUM_API_CALLS * out_channels, 1)

    def forward(self, adj, X):
        Z = self.gcn(adj, X)
        Z = F.relu(Z)
        Z = self.dropout(Z)
        Z = Z.reshape(Z.size(0), -1)
        out = self.fc(Z)
        return out.squeeze(1)

In [18]:
best_config = [
  0.6,
  128, #32
  10, #30
  31,
]

In [25]:
from sklearn.metrics import roc_auc_score

def train_fold(model, train_loader, val_loader, epochs):
  optimizer = torch.optim.Adam(
      model.parameters(),
      lr=0.001,
      betas=(0.9, 0.999)
  )

  criterion = nn.BCEWithLogitsLoss()

  for _ in range(epochs):
    print(f"Epoch #{_+1}...")
    model.train()
    for adj, X, labels in train_loader:
        adj, X, labels = adj.to(DEVICE), X.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(adj, X)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

  # Validation AUC
  model.eval()
  all_probs = []
  all_labels = []

  with torch.no_grad():
      for adj, X, labels in val_loader:
          adj, X = adj.to(DEVICE), X.to(DEVICE)
          logits = model(adj, X)
          probs = torch.sigmoid(logits)
          all_probs.extend(probs.cpu().numpy())
          all_labels.extend(labels.numpy())

  return roc_auc_score(all_labels, all_probs)

In [24]:
dropout, batch_size, epochs, out_channels = best_config

train_dataset = MalwareGraphDataset(train_df)
test_dataset = MalwareGraphDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

model = DGCNN(out_channels=out_channels, dropout=dropout).to(DEVICE)

train_fold(model, train_loader, test_loader, epochs)

Epoch #1...
Epoch #2...
Epoch #3...
Epoch #4...
Epoch #5...
Epoch #6...
Epoch #7...
Epoch #8...
Epoch #9...
Epoch #10...


np.float64(0.9695760842494752)

In [32]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for adj, X, labels in test_loader:
        adj, X = adj.to(DEVICE), X.to(DEVICE)
        logits = model(adj, X)
        probs = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

print(f"AUC-ROC: {roc_auc_score(all_labels, all_probs)}")
print(f"PR-AUC: {average_precision_score(all_labels,all_probs)}")

preds = [1 if x > 0.50 else 0 for x in all_probs]
print(classification_report(all_labels,preds))

AUC-ROC: 0.9695760842494752
PR-AUC: 0.9987799634701774
              precision    recall  f1-score   support

         0.0       0.87      0.53      0.66       324
         1.0       0.99      1.00      0.99     12839

    accuracy                           0.99     13163
   macro avg       0.93      0.76      0.82     13163
weighted avg       0.99      0.99      0.98     13163



In [33]:
torch.save(model.state_dict(), "dgcnn_best.pt")